In [1]:
import torch


logits=torch.tensor([2.0,1.0,0.0])
labels=["출력A","출력B","출력C"]

In [2]:
prob=torch.softmax(logits,dim=0)
assert torch.isclose(prob.sum(),torch.tensor(1.0),atol=1e-6)


In [3]:
temperature_results={str(t):torch.softmax(logits/t,dim=0).tolist() for t in (0.5,1.0,2.0)}

In [4]:
temperature_results

{'0.5': [0.8668133020401001, 0.11731041967868805, 0.01587623916566372],
 '1.0': [0.6652409434318542, 0.2447284758090973, 0.09003057330846786],
 '2.0': [0.5064803957939148, 0.30719590187072754, 0.18632373213768005]}

In [5]:
selected_index=int(torch.argmax(prob).item())
selected_token=labels[selected_index]

In [6]:
selected_token

'출력A'

In [7]:
sorted_p,sorted_ix=torch.sort(prob,descending=True)

In [13]:
sorted_p

tensor([0.6652, 0.2447, 0.0900])

In [8]:
csum=torch.cumsum(input=sorted_p,dim=0)
csum

tensor([0.6652, 0.9100, 1.0000])

In [9]:
remove=csum>0.75
remove

tensor([False,  True,  True])

In [10]:
remove[1:]=remove[:-1].clone()
remove[0]=False
remove

tensor([False, False,  True])

In [11]:
keep=~remove
keep

tensor([ True,  True, False])

In [12]:
kept_ix=sorted_ix[keep]
kept_p=sorted_p[keep]
kept_p

tensor([0.6652, 0.2447])

In [ ]:
top_p={
    "indices":kept_ix.tolist(),
    "labels":[labels[i] for i in kept_ix],
    "normalized_probs":(kept_p/kept_p.sum()).tolist()
}
top_p

{'indices': [0, 1],
 'labels': ['출력A', '출력B'],
 'normalized_probs': [0.7310585379600525, 0.2689414322376251]}

In [ ]:
# 확률이 높은 상위 k개 토큰만 후보로 남기고, 그 안에서 확률 합이 1이 되도록 다시 계산하는 작업입니다. 이를 Top-k 샘플링의 준비 과정

k=2
logits_vs=torch.tensor([2.0,1.0,0.0])
labels_vs=["출력A","출력B","출력C"]
prob_vs=torch.softmax(logits_vs,dim=0)
values_vs,indices_vs=torch.topk(input=prob_vs,k=k)
    # 해당 값들에 대해 정규화 수행
normalized_vs=(values_vs/values_vs.sum()).tolist()
top_k={"indices":indices_vs.tolist(),"labels":[labels[i] for i in indices_vs],"normalized_probs":normalized_vs}

print(logits_vs)
print(prob_vs)
print(normalized_vs)
print(top_k)

tensor([2., 1., 0.])
tensor([0.6652, 0.2447, 0.0900])
[0.7310585379600525, 0.2689414322376251]
{'indices': [0, 1], 'labels': ['출력A', '출력B'], 'normalized_probs': [0.7310585379600525, 0.2689414322376251]}


In [18]:
model=torch.nn.Linear(4,3,bias=False)
optimizer=torch.optim.SGD(model.parameters(),lr=0.1)

In [ ]:
x=torch.tensor([[1.,0.,0.,0.]])
target=torch.tensor([1])

In [19]:
model.weight

Parameter containing:
tensor([[-0.1383,  0.1453, -0.2954, -0.2530],
        [ 0.1074, -0.1236,  0.2906,  0.4449],
        [-0.4517,  0.4618, -0.4471,  0.3270]], requires_grad=True)

In [20]:
old_weight=model.weight.detach().clone()
old_weight


tensor([[-0.1383,  0.1453, -0.2954, -0.2530],
        [ 0.1074, -0.1236,  0.2906,  0.4449],
        [-0.4517,  0.4618, -0.4471,  0.3270]])

In [ ]:

    
    optimizer.zero_grad()
    loss=F.cross_entropy(model(x),target)
    loss.backward()
    optimizer.step()
    changed=not torch.allclose(old_weight,model.weight.detach())
    assert changed
    with torch.no_grad():
        eval_scores=model(x)
    training_check={"train_loss":float(loss.item()),"weights_changed":changed,
                    "inference_scores":eval_scores.squeeze(0).tolist()}
    # 해설: 이 모델은 next-token 대형 Transformer가 아닌 학습/추론 구분용 Linear 장난감입니다.

In [21]:
SPECIAL = ["[PAD]", "[UNK]", "[BOS]", "[EOS]"]
wordset = ['실습', '파일은', '개인', '폴더에', '보관합니다', '.']
vocab={tok:i for i,tok in enumerate(SPECIAL + wordset)}

vocab


{'[PAD]': 0,
 '[UNK]': 1,
 '[BOS]': 2,
 '[EOS]': 3,
 '실습': 4,
 '파일은': 5,
 '개인': 6,
 '폴더에': 7,
 '보관합니다': 8,
 '.': 9}

In [26]:

import re

corpus=["실습 파일은 개인 폴더에 보관합니다.", "프로젝트 폴더에서 서버를 실행합니다."]

def words(text: str) -> list[str]:
    """교육용: 한글 연속 글자/영문/숫자/문장부호 분리. 서브워드 토크나이저가 아님."""
    return re.findall(r'[가-힣]+|[A-Za-z]+|\d+|[^\w\s]', text)

a = {word for line in corpus for word in words(line)}
b = sorted(a)

print(a)
print(b)

{'보관합니다', '서버를', '파일은', '폴더에', '프로젝트', '실행합니다', '실습', '.', '폴더에서', '개인'}
['.', '개인', '보관합니다', '서버를', '실습', '실행합니다', '파일은', '폴더에', '폴더에서', '프로젝트']
